# Introduction

This project analyses eye-tracking data collected during a 3rd-year undergraduate neuroscience practical course (supervised by Pr. Aarabi). The goal is to study how different facial emotional stimuli influence visual exploration patterns across predefined regions of interest (ROIs).

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv('data_GR37.csv')
print('Setup complete !')

In [ ]:
print(df.shape)
df.head(11) # Visualising the whole dataframe

# Data cleaning

Firstly, I swapped the rows and columns as it is more conventional and to make the rest of the work easier. Secondly, I chose to split the main .csv into two dataframe, each containing fixation count or fixation  duration data. It aims to keep more simples dataframes and simplify future operations on them.

In [ ]:
# Deletion of empty rows and ligns and 'All recordings'

df.drop(['Unnamed: 0','Unnamed: 1'], axis=1, inplace=True)
df.drop([0,1,5,6,7], inplace=True)
df.drop([4,10], inplace=True)

# Empty values replacement

df = df.replace(['-', 'NaN', 'null'], '', regex=True)

# Transposition

df_transposed = df.T
df_transposed.head(10)

In [ ]:
# Correctly labeling rows and index

df_transposed.columns = ['Fixation count ID', 'Count value', 'Fixation duration ID', 'Duration value']
df_transposed = df_transposed.reset_index(drop=True)

print(df_transposed)

In [ ]:
# Seperating count and duration values from the original dataframe

df_count = df_transposed[['Fixation count ID', 'Count value']]
df_duration = df_transposed[['Fixation duration ID', 'Duration value']]

# Comma replacement

df_duration['Duration value'] = (
    df_duration['Duration value']
    .astype(str)
    .str.replace(',', '.')
    .str.strip()
)

# Converting to float

df_duration['Duration value'] = pd.to_numeric(df_duration['Duration value'], errors='coerce')
df_count['Count value'] = pd.to_numeric(df_count['Count value'], errors='coerce')

In [ ]:
print(df_count)
print(df_duration)

# Data treatment

We're adding additional columns for the different emotions submitted and the region of interest according to the following guidelines :

Emotions :
- Neutral: all files ending in …NES.JPG
- Happy: all files ending in …HAS.JPG
- Angry: all files ending in …ANS.JPG
- Sad: all files ending in … SAS.JPG

Regions of interest:
- Right eye: ROI 1, 5, 9, 13, 17, 21, 25, 29, 33, 37, 41, 45, 49, 53, 57, 61, 65, 69, 73, 77, 81, 85, 89, 93, 97, 101, 105, 109, 113, 117, 121, 125
- Left eye: ROI 2, 6, 10, 14, 18, 22, 26, 30, 34, 38, 42, 46, 50, 54, 58, 62, 66, 70, 74, 78, 82, 86, 90, 94, 98, 102, 106, 110, 114, 118, 122, 126
- Mouth: ROI 3, 7, 11, 15, 19, 23, 27, 31, 35, 39, 43, 47, 51, 55, 59, 63, 67, 71, 75, 79, 83, 87, 91, 95, 99, 103, 107, 111, 115, 119, 123, 127
- Nose: ROI 4, 8, 12, 16, 20, 24, 28, 32, 36, 40, 44, 48, 52, 56, 60, 64, 68, 72, 76, 80, 84, 88, 92, 96, 100, 104, 108, 112, 116, 120, 124, 128
- Area between the right eye and the left eye: R1, …, R32

In [ ]:
# Adding the additionnal rows to each dataframe

df_count.insert(loc=2, column='Emotion', value=0)
df_count.insert(loc=3, column='ROI', value=0)

df_duration.insert(loc=2, column='Emotion', value=0)
df_duration.insert(loc=3, column='ROI', value=0)

In [ ]:
print(df_count)
print(df_duration)

In [ ]:
# Emotions treatment function

def df_emotion_treatment(df):

    col_id = df.columns[0]

    conditions = [
        df[col_id].str.contains('NES', case=False, na=False),
        df[col_id].str.contains('HAS', case=False, na=False),
        df[col_id].str.contains('ANS', case=False, na=False),
        df[col_id].str.contains('SAS', case=False, na=False)
    ]
    emotions = ['Neutral', 'Happy', 'Angry', 'Sad']
    
    df['Emotion'] = np.select(conditions, emotions, default='Unknown')
    
    return df

In [ ]:
df_count = df_emotion_treatment(df_count)
df_duration = df_emotion_treatment(df_duration)

print(df_count)
print(df_duration)

In [ ]:
# ROI treatment

right_eye = [1, 5, 9, 13, 17, 21, 25, 29, 33, 37, 41, 45, 49, 53, 57, 61, 65, 69, 73, 77, 81, 85, 89, 93, 97, 101, 105, 109, 113, 117, 121, 125]
left_eye = [2, 6, 10, 14, 18, 22, 26, 30, 34, 38, 42, 46, 50, 54, 58, 62, 66, 70, 74, 78, 82, 86, 90, 94, 98, 102, 106, 110, 114, 118, 122, 126]
mouth = [3, 7, 11, 15, 19, 23, 27, 31, 35, 39, 43, 47, 51, 55, 59, 63, 67, 71, 75, 79, 83, 87, 91, 95, 99, 103, 107, 111, 115, 119, 123, 127]
nose = [4, 8, 12, 16, 20, 24, 28, 32, 36, 40, 44, 48, 52, 56, 60, 64, 68, 72, 76, 80, 84, 88, 92, 96, 100, 104, 108, 112, 116, 120, 124, 128]

def df_ROI_treatment(df):

    col_id = df.columns[0]
    
    roi_extraction = df[col_id].str.extract(r'_ROI(\d+)')
    num_roi = pd.to_numeric(roi_extraction[0])
    
    conditions_roi = [
        num_roi.isin(right_eye),
        num_roi.isin(left_eye),
        num_roi.isin(mouth),
        num_roi.isin(nose),
        df[col_id].str.contains(r'_R\d+_', regex=True) # MODIF
    ]
    
    noms_roi = ['Right eye', 'Left eye', 'Mouth', 'Nose', 'Area between eyes']
    df['ROI'] = np.select(conditions_roi, noms_roi, default='Unknown')

    return df

In [ ]:
df_count = df_ROI_treatment(df_count)
df_duration = df_ROI_treatment(df_duration)

In [ ]:
print(df_count)

In [ ]:
print(df_duration)

# Data analysis

## Statistic description

In [ ]:
print(df_count.head(5))

type(df_duration['Duration value'])

In [ ]:
stats_duration = (
    df_duration.groupby(['Emotion', 'ROI'])['Duration value']
    .agg(['mean', 'std', 'sum'])
    .reset_index()
)

stats_count = (
    df_count.groupby(['Emotion', 'ROI'])['Count value']
    .agg(['mean', 'std', 'sum'])
    .reset_index()
)

In [ ]:
print(stats_duration)

In [ ]:
print(stats_count)

## Data visualisation and interpretation

## Fixation duration interpretation

The eyes drew the most attention from viewers, especially the right eye (mean duration between 0.46 and 0.57) and the left eye for sad faces (around 0.51). The mouth holds attention for particular emotions : fixations are longer while observing angry (0.45) and sad (0.40) faces compared to the other non-eye areas. Additionally, The nose has the shortest fixation duration for nearly all emotions (between 0.24 and 0.32).   

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Fixation duration heatmap

pivot_duration = df_duration.pivot_table(
    index='ROI', columns='Emotion', values='Duration value', aggfunc='mean'
)
plt.figure(figsize=(8, 5))
sns.heatmap(pivot_duration, annot=True, cmap='Blues', fmt='.2f')
plt.title('Mean fixation time (ms) by ROI and emotion')
plt.show()

In [ ]:
# Fixation duration barplot

plt.figure(figsize=(10, 6))
sns.barplot(
    data=df_duration,
    x='ROI',
    y='Duration value',
    hue='Emotion',
)
plt.title('Fixation duration depending on ROI and emotion')
plt.ylabel('Duration (ms)')
plt.show()

## Fixation count interpretation

Viewers typically focus more on the right eye (averaging between 4.00 and 4.88 fixations), making it the most looked area. The nose has the lowest count overall (around 1.00 to 1.83 fixations on average), meaning people barely look at it. A moderate amount of fixations (usually between 1.3 and 3.1) are given to the mouth and left eye.  

In [ ]:
# Fixation count heatmap

pivot_count = df_count.pivot_table(
    index='ROI', columns='Emotion', values='Count value', aggfunc='mean'
)
plt.figure(figsize=(8, 5))
sns.heatmap(pivot_count, annot=True, cmap='Blues', fmt='.2f')
plt.title('Mean fixation count by ROI and emotion')
plt.show()

In [ ]:
# Fixation count barplot

plt.figure(figsize=(10, 6))
sns.barplot(
    data=df_count,
    x='ROI',
    y='Count value',
    hue='Emotion',
)
plt.title('Fixation count depending on ROI and emotion')
plt.ylabel('Count')
plt.show()

## Interpretation summary

Overall, the eye area continues to be the primary focal point in both count and duration, but the mouth becomes significant when viewers try to interpret negative emotions like anger or sadness. In contrast, regions like the nose and the area between the eyes receive minimal attention regardless of the emotion displayed.

The overlapping error bars visible on the barplots show noticeable variability between trials, particularly for fixation durations. Combining datasets from all groups in the future would enhance the sample size and help lower these standard deviations, as this research only includes data from my group for now.